In [1]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import battery_utility_calculator as buc
from tqdm.notebook import tqdm


In [2]:
con = sqlite3.connect("./mas4te-assume.db")

In [3]:
SIMULATION = 'mas4te_simulation_47weeks_profit_4UC'
SIM_FILTER = f"simulation = '{SIMULATION}'"

In [4]:
def tslice(df, start, end):
    return df[(df["datetime"] >= start) & (df["datetime"] <= end)].reset_index(drop=True)

In [5]:
sql = f"""
SELECT 
    min(start_time) as earliest,
    max(start_time) as latest,
    count(distinct start_time) as n_timesteps
FROM market_orders
WHERE {SIM_FILTER}
"""
print(pd.read_sql(sql, con))

                     earliest                      latest  n_timesteps
0  2023-01-09 01:00:00.000000  2023-12-11 01:00:00.000000           49


In [6]:
sql = "PRAGMA table_info(market_orders)"
print(pd.read_sql(sql, con))

    cid             name      type  notnull dflt_value  pk
0     0       start_time  DATETIME        0       None   0
1     1   accepted_price     FLOAT        0       None   0
2     2  accepted_volume     FLOAT        0       None   0
3     3           bid_id    BIGINT        0       None   0
4     4         bid_type      TEXT        0       None   0
5     5           c_rate    BIGINT        0       None   0
6     6         end_time  DATETIME        0       None   0
7     7        market_id      TEXT        0       None   0
8     8             node      TEXT        0       None   0
9     9            price     FLOAT        0       None   0
10   10       simulation      TEXT        0       None   0
11   11          unit_id      TEXT        0       None   0
12   12           volume     FLOAT        0       None   0


In [7]:
sql = "PRAGMA table_info(volumes_worth)"
print(pd.read_sql(sql, con))

Empty DataFrame
Columns: [cid, name, type, notnull, dflt_value, pk]
Index: []


In [8]:
# demand = pd.read_hdf("./../../simulation_data/demand.h5")
# demand["datetime"] = pd.to_datetime(demand["datetime"])
# demand.head()

demand = pd.read_csv("./example_data/demand.csv")
demand.index = pd.to_datetime(demand.index)
demand.index.name = "datetime"
demand.head()


,Unnamed: 0,demand,demand_0,demand_1,demand_2,demand_3,demand_4,demand_5,demand_6,demand_7,...,demand_20,demand_21,demand_22,demand_23,demand_24,demand_25,demand_26,demand_27,demand_28,demand_29
datetime,,,,,,,,,,,,,,,,,,,,,
1970-01-01 00:00:00.000000000,2022-12-31 23:00:00,0.328460,0.330936,0.220033,0.463716,0.459913,0.343089,0.186142,0.275083,0.294386,...,0.238201,0.335960,0.195029,0.462649,0.280858,0.171950,0.285558,0.348381,0.228714,0.372016
1970-01-01 00:00:00.000000001,2023-01-01 00:00:00,0.245473,0.279050,0.137174,0.221312,0.283852,0.296512,0.208496,0.141505,0.251460,...,0.201897,0.156751,0.141891,0.190808,0.229025,0.150142,0.218590,0.278285,0.347520,0.194985
1970-01-01 00:00:00.000000002,2023-01-01 01:00:00,0.241935,0.170133,0.361851,0.309354,0.172840,0.134966,0.279717,0.281897,0.133311,...,0.328134,0.281805,0.301392,0.327850,0.346325,0.305124,0.194843,0.288599,0.189038,0.301150
1970-01-01 00:00:00.000000003,2023-01-01 02:00:00,0.226964,0.245833,0.214941,0.332301,0.329056,0.208223,0.223870,0.114232,0.248359,...,0.249036,0.221807,0.239104,0.219037,0.292358,0.258821,0.260991,0.241027,0.229693,0.211790
1970-01-01 00:00:00.000000004,2023-01-01 03:00:00,0.228801,0.153725,0.122101,0.326221,0.246757,0.175038,0.304277,0.265360,0.277222,...,0.257739,0.286397,0.173339,0.297886,0.291429,0.172003,0.328290,0.193009,0.175165,0.138781


In [9]:
# solar = pd.read_csv("./../../simulation_data/solar_generation_aachen_2023.csv", index_col=0)
# solar["datetime"] = pd.to_datetime(solar["datetime"])
# solar.head()

solar = pd.read_csv("./example_data/solar.csv", index_col=0)
solar.index = pd.to_datetime(solar.index)
solar.index.name = "datetime"  # optional, just for clarity
solar.head()

,solar,solar_0,solar_1,solar_2,solar_3,solar_4,solar_5,solar_6,solar_7,solar_8,...,solar_20,solar_21,solar_22,solar_23,solar_24,solar_25,solar_26,solar_27,solar_28,solar_29
datetime,,,,,,,,,,,,,,,,,,,,,
2022-12-31 23:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-01-01 03:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
# px.line(solar, "datetime", "solar")

px.line(solar.reset_index(), x="datetime", y="solar")


In [11]:
# prices = pd.read_csv("./../../simulation_data/prices.csv", index_col=0)
# prices["datetime"] = pd.to_datetime(prices["datetime"])
# prices.head()

prices = pd.read_csv("./example_data/prices.csv", index_col=0)
prices.index = pd.to_datetime(prices.index)
prices.index.name = "datetime"
prices.head()

,wholesale,eeg,community,grid
datetime,,,,
2022-12-31 23:00:00,-0.00517,0.07,0,0.136734
2023-01-01 00:00:00,-0.00107,0.07,0,0.129530
2023-01-01 01:00:00,-0.00147,0.07,0,0.120096
2023-01-01 02:00:00,-0.00508,0.07,0,0.115365
2023-01-01 03:00:00,-0.00449,0.07,0,0.108565


# Bidding curves

In [12]:
sql = """
    SELECT distinct(unit_id)
    FROM market_orders
    WHERE volume > 0
    AND start_time = '2023-01-30 01:00:00.000000'
"""
sellers = pd.read_sql(sql, con)
seller_bcs = {}
for seller in tqdm(sellers["unit_id"].unique()):
    volumes = pd.read_sql(f"select * from volumes_worth where unit_id = '{seller}' and product_start = '2023-07-31 01:00:00.000000'", con)
    bc = buc.calculate_bidding_curve(volumes, "seller")
    seller_bcs[seller] = bc


sql = """
    SELECT distinct(unit_id)
    FROM market_orders
    WHERE volume < 0
    AND start_time = '2023-01-01 01:00:00.000000'
"""
buyers = pd.read_sql(sql, con)
buyer_bcs = {}
for buyer in tqdm(buyers["unit_id"].unique()):
    print('buyer ', buyer)
    volumes = pd.read_sql(f"select * from volumes_worth where unit_id = '{buyer}' and product_start = '2023-07-31 01:00:00.000000'", con)
    bc = buc.calculate_bidding_curve(volumes, "buyer")
    buyer_bcs[buyer] = bc




  0%|          | 0/2 [00:00<?, ?it/s]

DatabaseError: Execution failed on sql 'select * from volumes_worth where unit_id = 'S_01' and product_start = '2023-07-31 01:00:00.000000'': no such table: volumes_worth

In [ ]:
d = pd.concat(buyer_bcs.values()).sort_values("marginal_price_per_kwh", ascending=False, ignore_index=True)
d["cum_vol"] = d["volume"].cumsum()
d = d[d["cum_vol"] <= 500]

s = pd.concat(seller_bcs.values()).sort_values("marginal_price_per_kwh", ascending=True, ignore_index=True)
s["cum_vol"] = s["volume"].cumsum()

fig = go.Figure()
fig.add_scatter(x=d["cum_vol"], y=d["marginal_price_per_kwh"], name="Consumers")
fig.add_scatter(x=s["cum_vol"], y=s["marginal_price_per_kwh"], name="Providers")
fig.update_layout(xaxis_title="Cumulative volume in kWh", yaxis_title="Marginal price in €/kWh")
fig.update_yaxes(rangemode="tozero")
# fig.write_image("./images/combined_bidding_curves.pdf")
fig.show()

NameError: name 'buyer_bcs' is not defined

# Prices / Welfare over time

In [12]:
sql = f"""
SELECT
	start_time,
	sum(volume) as bid_volume,
	sum(accepted_volume) as accepted_volume,
	avg(price) as avg_bid_price,
	avg(accepted_price) * 100 as clearing_price,
	sum(abs(accepted_price-price)) as welfare
FROM market_orders
WHERE {SIM_FILTER}
GROUP BY start_time
"""
prices_over_time = pd.read_sql(sql, con)
fig = px.line(prices_over_time, "start_time", "clearing_price")
fig.update_layout(
    title="Clearing price over time",
    xaxis_title="Time",
    yaxis_title="Clearing price in ct. / kWh")
fig.update_yaxes(rangemode="tozero")

In [13]:
sql = f"""
SELECT
	start_time,
	sum(volume) as bid_volume,
	sum(accepted_volume) as accepted_volume,
	avg(price) as avg_bid_price,
	avg(accepted_price) as clearing_price,
	sum(abs(accepted_price-price)) as welfare,
    sum(abs(accepted_price-price)) / count(distinct(unit_id)) as mean_welfare
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY start_time
"""
welfare_over_time = pd.read_sql(sql, con)
fig = px.line(welfare_over_time, "start_time", "welfare")
fig.update_layout(
    title="Welfare over time",
    xaxis_title="Time",
    yaxis_title="Total welfare in €"
)

In [14]:
sql = f"""
SELECT
	start_time,
	sum(volume) as bid_volume,
	sum(accepted_volume) as accepted_volume,
	avg(price) as avg_bid_price,
	avg(accepted_price) as clearing_price,
	sum(abs(accepted_price-price)) as welfare,
    sum(abs(accepted_price-price)) / count(distinct(unit_id)) as avg_welfare
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY start_time
"""
prices_over_time = pd.read_sql(sql, con)
px.line(prices_over_time, "start_time", "avg_welfare").update_layout(title="Average welfare over time")

In [15]:
sql = f"""
SELECT
	start_time,
    unit_id,
	sum(volume) as bid_volume,
	sum(accepted_volume) as accepted_volume,
	avg(price) as avg_bid_price,
	avg(accepted_price) as clearing_price,
	sum(abs(accepted_price-price)) as welfare
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY start_time, unit_id
"""
prices_over_time = pd.read_sql(sql, con)
px.line(prices_over_time, "start_time", "welfare", color="unit_id").update_layout(title="Mean welfare over time")

In [16]:
sql = f"""
SELECT
    start_time,
    unit_id,
    sum(abs(accepted_volume)) as traded_volume
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY start_time, unit_id
"""
volume_over_time = pd.read_sql(sql, con)
px.line(volume_over_time, "start_time", "traded_volume", color="unit_id").update_layout(
    title="Traded volume over time per unit",
    xaxis_title="Time",
    yaxis_title="Traded volume in kWh"
)

In [17]:
sql = f"""
SELECT
    unit_id,
    sum(abs(accepted_price - price) * abs(accepted_volume)) as total_welfare
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY unit_id
"""
welfare_per_unit = pd.read_sql(sql, con)
print(welfare_per_unit.to_string(index=False))
print('')

sql = f"""
SELECT
    unit_id,
    sum(abs(accepted_volume)) as total_volume
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY unit_id
"""
volume_per_unit = pd.read_sql(sql, con)
print(volume_per_unit.to_string(index=False))

print('')

sql = f"""
SELECT
    unit_id,
    avg(accepted_price - price) as avg_price_diff
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY unit_id
"""
price_diff_per_unit = pd.read_sql(sql, con)
print(price_diff_per_unit.to_string(index=False))




unit_id  total_welfare
   B_01       7.840957
   B_02      10.500120
   S_01       0.144532
   S_02       4.108209

unit_id  total_volume
   B_01          63.0
   B_02          54.0
   S_01          63.0
   S_02          24.0

unit_id  avg_price_diff
   B_01        0.038861
   B_02       -0.176973
   S_01        0.002294
   S_02        0.171175


In [30]:
sql = f"""
SELECT
    unit_id,
    sum(abs(accepted_price - price) * abs(accepted_volume)) as total_welfare,
    sum(abs(accepted_volume)) as traded_volume,
    avg(accepted_price - price) as avg_price_diff,
    sum(abs(accepted_price) * abs(accepted_volume)) as total_money_traded
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY unit_id
"""
summary_per_unit = pd.read_sql(sql, con)

totals = summary_per_unit.sum(numeric_only=True)
totals["unit_id"] = "TOTAL"
totals["avg_price_diff"] = summary_per_unit["avg_price_diff"].mean()
totals["traded_volume"] = summary_per_unit["traded_volume"].sum() / 2  # avoid double counting
summary_per_unit = pd.concat([summary_per_unit, totals.to_frame().T], ignore_index=True)

print(summary_per_unit.to_string(index=False))

unit_id total_welfare traded_volume avg_price_diff total_money_traded
   B_01      7.840957          63.0       0.038861           9.588138
   B_02      10.50012          54.0      -0.176973           8.053144
   S_01      0.144532          63.0       0.002294           9.588138
   S_02      4.108209          24.0       0.171175           4.738237
  TOTAL     22.593818         102.0       0.008839          31.967657


In [32]:
sql = f"""
SELECT
    unit_id,
    start_time,
    count(*) as n_rows,
    sum(abs(accepted_volume)) as volume
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY unit_id, start_time
ORDER BY start_time
LIMIT 20
"""
print(pd.read_sql(sql, con).to_string(index=False))

unit_id                 start_time  n_rows  volume
   B_01 2023-01-09 01:00:00.000000       4     4.0
   B_02 2023-01-09 01:00:00.000000       3     3.0
   S_01 2023-01-09 01:00:00.000000       4     4.0
   S_02 2023-01-09 01:00:00.000000       4     4.0
   B_01 2023-01-16 01:00:00.000000       4     4.0
   B_02 2023-01-16 01:00:00.000000       3     3.0
   S_01 2023-01-16 01:00:00.000000       4     4.0
   B_01 2023-01-23 01:00:00.000000       3     3.0
   B_02 2023-01-23 01:00:00.000000       3     3.0
   S_01 2023-01-23 01:00:00.000000       3     3.0
   B_01 2023-01-30 01:00:00.000000       2     2.0
   B_02 2023-01-30 01:00:00.000000       1     1.0
   S_01 2023-01-30 01:00:00.000000       2     2.0
   S_02 2023-01-30 01:00:00.000000       2     2.0
   B_01 2023-02-06 01:00:00.000000       3     3.0
   B_02 2023-02-06 01:00:00.000000       3     3.0
   S_01 2023-02-06 01:00:00.000000       3     3.0
   B_01 2023-02-20 01:00:00.000000       3     3.0
   B_02 2023-02-20 01:00:00.000

In [19]:
sql = f"""
    SELECT
        avg(accepted_price) AS clearing_price
    FROM market_orders
    WHERE start_time >= '2023-04-01'
    AND start_time <= '2023-10-01'
    AND {SIM_FILTER}
"""
cp_summer = pd.read_sql(sql, con)
sql = f"""
    SELECT
        avg(accepted_price) AS clearing_price
    FROM market_orders
    WHERE start_time < '2023-04-01'
    OR start_time > '2023-10-01'
    AND {SIM_FILTER}
"""
cp_winter = pd.read_sql(sql, con)
fig = go.Figure()
fig.add_bar(x=["Winter"], y=cp_winter["clearing_price"])
fig.add_bar(x=["Summer"], y=cp_summer["clearing_price"])
fig.update_layout(
    showlegend=False,
    title="Average clearing price in summer / winter",
    yaxis_title="Clearing price in €/kWh")

# Price / Welfare vs seller / buyer, summer / winter

In [20]:
sql = f"""
    SELECT sum(price - accepted_price) as welfare
    FROM market_orders
    WHERE accepted_volume < 0
    AND start_time >= '2023-04-01'
    AND start_time <= '2023-10-01'
    AND {SIM_FILTER}
"""
buyer_welfare_summer = pd.read_sql(sql, con)
sql = f"""
    SELECT sum(price - accepted_price) as welfare
    FROM market_orders
    WHERE accepted_volume < 0
    AND (start_time < '2023-04-01'
    OR start_time > '2023-10-01')
    AND {SIM_FILTER}
"""
buyer_welfare_winter = pd.read_sql(sql, con)

sql = f"""
    SELECT sum(accepted_price - price) as welfare
    FROM market_orders
    WHERE accepted_volume > 0
    AND start_time >= '2023-04-01'
    AND start_time <= '2023-10-01'
    AND {SIM_FILTER}
"""
seller_welfare_summer = pd.read_sql(sql, con)
sql = f"""
    SELECT sum(accepted_price - price) as welfare
    FROM market_orders
    WHERE accepted_volume > 0
    AND (start_time < '2023-04-01'
    OR start_time > '2023-10-01')
    AND {SIM_FILTER}
"""
seller_welfare_winter = pd.read_sql(sql, con)
df = pd.concat([buyer_welfare_summer, buyer_welfare_winter, seller_welfare_summer, seller_welfare_winter])
df["timeofyear"] = ["Summer", "Winter", "Summer", "Winter"]
df["Type"] = ["Buyer", "Buyer", "Seller", "Seller"]

fig = px.bar(df, "timeofyear", "welfare", color="Type", barmode="group")
fig.update_layout(xaxis_title="", yaxis_title="Cumulative welfare in €")

In [21]:
# buyer
sql = f"""
    SELECT
        sum(price - accepted_price) / count(distinct(unit_id)) as welfare
    FROM market_orders
    WHERE accepted_volume < 0
    AND start_time >= '2023-04-01'
    AND start_time <= '2023-10-01'
    AND {SIM_FILTER}
"""
buyer_welfare_summer = pd.read_sql(sql, con)
sql = f"""
    SELECT sum(price - accepted_price) / count(distinct(unit_id)) as welfare
    FROM market_orders
    WHERE accepted_volume < 0
    AND (start_time < '2023-04-01'
    OR start_time > '2023-10-01')
    AND {SIM_FILTER}
"""
buyer_welfare_winter = pd.read_sql(sql, con)


# seller
sql = f"""
    SELECT sum(accepted_price - price) / count(distinct(unit_id)) as welfare
    FROM market_orders
    WHERE accepted_volume > 0
    AND start_time >= '2023-04-01'
    AND start_time <= '2023-10-01'
    AND {SIM_FILTER}
"""
seller_welfare_summer = pd.read_sql(sql, con)
sql = f"""
    SELECT sum(accepted_price - price) / count(distinct(unit_id)) as welfare
    FROM market_orders
    WHERE accepted_volume > 0
    AND (start_time < '2023-04-01'
    OR start_time > '2023-10-01')
    AND {SIM_FILTER}
"""
seller_welfare_winter = pd.read_sql(sql, con)

df = pd.concat([buyer_welfare_summer, buyer_welfare_winter, seller_welfare_summer, seller_welfare_winter])
df["timeofyear"] = ["Summer", "Winter", "Summer", "Winter"]
df["Participant"] = ["Buyer", "Buyer", "Seller", "Seller"]
df["welfare"] = df["welfare"].round(2)

fig = px.bar(df, "timeofyear", "welfare", color="Participant", barmode="group", text="welfare")
fig.update_layout(xaxis_title="", yaxis_title="Average welfare in €")

In [22]:
sql = f"""
    SELECT sum(price - accepted_price) as welfare
    FROM market_orders
    WHERE accepted_volume < 0
    AND start_time >= '2023-04-01'
    AND start_time <= '2023-10-01'
    AND {SIM_FILTER}
    GROUP BY unit_id
"""
buyer_welfare_summer = pd.read_sql(sql, con)
buyer_welfare_summer["timeofyear"] = "Summer"
buyer_welfare_summer["Participant"] = "Buyer"
sql = f"""
    SELECT sum(price - accepted_price) as welfare
    FROM market_orders
    WHERE accepted_volume < 0
    AND ( start_time < '2023-04-01'
    OR start_time > '2023-10-01')
    AND {SIM_FILTER}
    GROUP BY unit_id
"""
buyer_welfare_winter = pd.read_sql(sql, con)
buyer_welfare_winter["timeofyear"] = "Winter"
buyer_welfare_winter["Participant"] = "Buyer"
sql = f"""
    SELECT sum(accepted_price - price) as welfare
    FROM market_orders
    WHERE accepted_volume > 0
    AND start_time >= '2023-04-01'
    AND start_time <= '2023-10-01'
    AND {SIM_FILTER}
    GROUP BY unit_id
"""
seller_welfare_summer = pd.read_sql(sql, con)
seller_welfare_summer["timeofyear"] = "Summer"
seller_welfare_summer["Participant"] = "Seller"
sql = f"""
    SELECT sum(accepted_price - price) as welfare
    FROM market_orders
    WHERE accepted_volume > 0
    AND ( start_time < '2023-04-01'
    OR start_time > '2023-10-01')
    AND {SIM_FILTER}
    GROUP BY unit_id
"""
seller_welfare_winter = pd.read_sql(sql, con)
seller_welfare_winter["timeofyear"] = "Winter"
seller_welfare_winter["Participant"] = "Seller"

df = pd.concat([buyer_welfare_summer, buyer_welfare_winter, seller_welfare_summer, seller_welfare_winter])
# df["timeofyear"] = ["Summer", "Winter", "Summer", "Winter"]
# df["Participant"] = ["Buyer", "Buyer", "Seller", "Seller"]
df["welfare"] = df["welfare"].round(2)
fig = px.box(df, "timeofyear", "welfare", color="Participant")#
fig.add_annotation(
    x="Summer",
    text="Median welfare summer <br> Buyer: 0.16€ <br> Seller: 9.30€",
    xshift=-10,
    yshift=100,
)
fig.add_annotation(
    x="Winter",
    text="Median welfare winter <br> Buyer: 0.04€ <br> Seller: 24.70€",
    xshift=-10,
    yshift=125,
)
fig.update_layout(
    yaxis_title="Welfare per participant in €",
    xaxis_title=""
)

# Volumes

In [23]:
sql = f"""
SELECT
	sum(accepted_volume) as total_traded_volume
FROM market_orders
WHERE accepted_volume >= 0
AND {SIM_FILTER}
"""
pd.read_sql(sql, con)

,total_traded_volume
0,70.0


In [23]:
sql = f"""
SELECT
	start_time,
	sum(accepted_volume) as traded_volume
FROM market_orders
WHERE accepted_volume >= 0
AND {SIM_FILTER}
GROUP BY start_time
"""
acc_vol_over_time = pd.read_sql(sql, con)
fig = px.line(acc_vol_over_time, "start_time", "traded_volume")
fig.update_layout(xaxis_title="Time", yaxis_title="Traded volume in kWh")
fig.update_yaxes(rangemode="tozero")

In [25]:
sql = f"""
SELECT
	start_time,
	sum(volume) as rejected_volume
FROM market_orders
WHERE volume >= 0
AND accepted_volume = 0
AND {SIM_FILTER}
GROUP BY start_time
"""
rej_vol_over_time = pd.read_sql(sql, con)
fig = px.line(rej_vol_over_time, "start_time", "rejected_volume")
fig.update_layout(title="Rejected volume over time")
fig.update_yaxes(rangemode="tozero")

In [24]:
sql = f"""
SELECT
	start_time,
    unit_id,
	sum(abs(volume)) as volume
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY start_time, unit_id
ORDER BY start_time ASC
"""
rej_vol_over_time = pd.read_sql(sql, con)
fig = px.line(rej_vol_over_time, "start_time", "volume", color="unit_id")
fig.update_layout(title="Traded volume over time per unit")
fig.update_yaxes(rangemode="tozero")

In [27]:
sql = f"""
SELECT
    unit_id,
	sum(abs(accepted_volume)) as total_traded_volume
FROM market_orders
WHERE accepted_volume != 0
AND {SIM_FILTER}
GROUP BY unit_id
"""
df = pd.read_sql(sql, con)
px.bar(df, "unit_id", "total_traded_volume")

In [25]:
sql = f"""
SELECT
    unit_id,
    CASE WHEN volume > 0 THEN 'Seller' ELSE 'Buyer' END as role,
    count(*) as total_bids,
    sum(CASE WHEN accepted_volume != 0 THEN 1 ELSE 0 END) as matched_bids,
    round(100.0 * sum(CASE WHEN accepted_volume != 0 THEN 1 ELSE 0 END) / count(*), 2) as match_rate_pct
FROM market_orders
WHERE {SIM_FILTER}
GROUP BY unit_id, role
"""
match_rate = pd.read_sql(sql, con)

totals = match_rate.groupby("role").sum(numeric_only=True).reset_index()
totals["unit_id"] = "TOTAL"
totals["match_rate_pct"] = round(100.0 * totals["matched_bids"] / totals["total_bids"], 2)

overall = match_rate.sum(numeric_only=True)
overall["unit_id"] = "TOTAL"
overall["role"] = "TOTAL"
overall["match_rate_pct"] = round(100.0 * overall["matched_bids"] / overall["total_bids"], 2)

match_rate = pd.concat([match_rate, totals, overall.to_frame().T], ignore_index=True)
print(match_rate.to_string(index=False))

unit_id   role total_bids matched_bids match_rate_pct
   B_01  Buyer        182           63          34.62
   B_02  Buyer        147           54          36.73
   S_01 Seller        392           63          16.07
   S_02 Seller        196           24          12.24
  TOTAL  Buyer        329          117          35.56
  TOTAL Seller        588           87           14.8
  TOTAL  TOTAL      917.0        204.0          22.25


# Bid prices over time

In [26]:
sql = f"""
SELECT
    avg(price) as ask_price,
    start_time
FROM market_orders
WHERE volume > 0
--AND accepted_volume != 0
AND {SIM_FILTER}
GROUP BY start_time
"""
asks = pd.read_sql(sql, con)

sql = f"""
SELECT
    avg(price) as ask_price,
    start_time
FROM market_orders
WHERE volume < 0
--AND accepted_volume != 0
AND {SIM_FILTER}
GROUP BY start_time
"""
bids = pd.read_sql(sql, con)

fig = go.Figure()
fig.add_scatter(x=asks["start_time"], y=asks["ask_price"], name="Providers")
fig.add_scatter(x=bids["start_time"], y=bids["ask_price"], name="Consumers")

fig.update_layout(
    xaxis_title="Time",
    yaxis_title="Price (€/kWh)",
)

# Welfare distributions

In [27]:
sql = f"""
    select
        sum(abs(price - accepted_price)) as welfare,
        unit_id
    from market_orders
    where accepted_volume != 0
    AND {SIM_FILTER}
    group by unit_id
"""
df = pd.read_sql(sql, con)
px.histogram(df, "welfare")

# Interesting dates

In [26]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True)
start = "2023-06-26"
end = "2023-07-02 23:45:00"
s = tslice(solar, start, end)
p = tslice(prices, start, end)
d = tslice(demand, start, end)
d = d.groupby("datetime", as_index=False).sum()
fig.add_scatter(x=s["datetime"], y=s["solar"], name="Solar gen", row=1, col=1)
fig.add_scatter(x=p["datetime"], y=p["wholesale"], name="Wholesale prices", row=2, col=1)
fig.add_scatter(x=p["datetime"], y=p["supplier"], name="Supplier prices", row=2, col=1)
fig.add_scatter(x=d["datetime"], y=d["load_kw"], name="Demand", row=3, col=1)

KeyError: 'datetime'

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True)
start = "2023-01-23"
end = "2023-02-12 23:45:00"
s = tslice(solar, start, end)
p = tslice(prices, start, end)
d = tslice(demand, start, end)
d = d.groupby("datetime", as_index=False).sum()
fig.add_scatter(x=s["datetime"], y=s["solar"], name="Solar gen", row=1, col=1)
fig.add_scatter(x=p["datetime"], y=p["wholesale"], name="Wholesale prices", row=2, col=1)
fig.add_scatter(x=d["datetime"], y=d["load_kw"], name="Demand", row=3, col=1)
fig.add_vline(x="2023-01-30 00:00:00")
fig.add_vline(x="2023-02-06 00:00:00")

# Correlations

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=prices_over_time["start_time"], y=prices_over_time["clearing_price"], name="Clearing price")
tmp = prices.resample("W", on="datetime").mean().reset_index()
fig.add_bar(x=tmp["datetime"], y=tmp["wholesale"], name="Wholesale market price", secondary_y=True)

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=prices_over_time["start_time"], y=prices_over_time["clearing_price"], name="Clearing price")
tmp = solar.resample("W", on="datetime").mean().reset_index()
fig.add_bar(x=tmp["datetime"], y=tmp["solar"], name="Solar generation", secondary_y=True)

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=prices_over_time["start_time"], y=prices_over_time["clearing_price"], name="Clearing price")
tmp = solar.resample("W", on="datetime").mean().reset_index()
fig.add_scatter(x=acc_vol_over_time["start_time"], y=acc_vol_over_time["traded_volume"], name="Traded volume", secondary_y=True)
fig.update_yaxes(rangemode="tozero")

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=welfare_over_time["start_time"], y=welfare_over_time["welfare"], name="Welfare")
tmp = solar.resample("W", on="datetime").mean().reset_index()
fig.add_scatter(x=tmp["datetime"], y=tmp["solar"], name="Solar generation", secondary_y=True)
fig.update_layout(xaxis_title="Time")
fig.update_yaxes(title="Welfare in €")
fig.update_yaxes(title="Solar generation in %", secondary_y=True)
fig.update_yaxes(rangemode="tozero")

# Willingness to pay vs market clearing prices

In [27]:
sql = """
    SELECT
        avg(accepted_price) as clearing_price,
        avg(price) as bid_price,
        start_time
    FROM market_orders
    WHERE volume = -1
    AND accepted_volume = -1
    GROUP BY start_time
    ORDER BY start_time ASC
"""
bid_prices = pd.read_sql(sql, con)
sql = """
    SELECT
        avg(accepted_price) as clearing_price,
        avg(price) as bid_price,
        start_time
    FROM market_orders
    WHERE volume = 1
    AND accepted_volume = 1
    GROUP BY start_time
    ORDER BY start_time ASC
"""
ask_prices = pd.read_sql(sql, con)

fig = go.Figure()
fig.add_scatter(x=bid_prices["start_time"], y=bid_prices["bid_price"], name="Buyer")
fig.add_scatter(x=ask_prices["start_time"], y=ask_prices["bid_price"], name="Seller")
fig.add_scatter(x=bid_prices["start_time"], y=bid_prices["clearing_price"], name="Clearing price")

In [29]:
sql = """
    SELECT
        avg(worth) as bid_price,
        product_start as start_time
    FROM volumes_worth
    WHERE volume = 1
    AND worth > 0
    GROUP BY product_start
    ORDER BY product_start ASC
"""
bid_prices = pd.read_sql(sql, con)
sql = """
    WITH first_kwh_worth AS (
        SELECT
            unit_id,
            product_start,
            max(worth) as max_worth
        FROM volumes_worth
        WHERE worth < 0
        GROUP BY unit_id, product_start
    )
    SELECT abs(avg(max_worth)) as bid_price, product_start as start_time
    FROM first_kwh_worth
    GROUP BY product_start
"""
ask_prices = pd.read_sql(sql, con)
sql = """
    SELECT 
        avg(accepted_price) as clearing_price,
        start_time
    FROM market_orders
    WHERE accepted_volume != 0
    GROUP BY start_time
    ORDER BY start_time ASC
"""
mcp = pd.read_sql(sql, con)

fig = go.Figure()
fig.add_scatter(x=bid_prices["start_time"], y=bid_prices["bid_price"], name="Buyer")
fig.add_scatter(x=ask_prices["start_time"], y=ask_prices["bid_price"], name="Seller")
fig.add_scatter(x=mcp["start_time"], y=mcp["clearing_price"], name="Clearing price")

DatabaseError: Execution failed on sql '
    SELECT
        avg(worth) as bid_price,
        product_start as start_time
    FROM volumes_worth
    WHERE volume = 1
    AND worth > 0
    GROUP BY product_start
    ORDER BY product_start ASC
': no such table: volumes_worth

# Bid prices over time

In [30]:
sql = """
    SELECT avg(price) as avg_ask_summer
    FROM market_orders
    WHERE volume > 0
    AND accepted_volume != 0
    AND start_time >= '2023-04-01'
    AND start_time <= '2023-10-01'
"""
print(pd.read_sql(sql, con))
sql = """
    SELECT avg(price) as avg_bid_summer
    FROM market_orders
    WHERE volume < 0
    AND accepted_volume != 0
    AND start_time >= '2023-04-01'
    AND start_time <= '2023-10-01'
"""
print(pd.read_sql(sql, con))

sql = """
    SELECT avg(price) as avg_ask_winter
    FROM market_orders
    WHERE volume > 0
    AND accepted_volume != 0
    AND ( start_time < '2023-04-01'
    OR start_time > '2023-10-01')
"""
print(pd.read_sql(sql, con))
sql = """
    SELECT avg(price) as avg_bid_winter
    FROM market_orders
    WHERE volume < 0
    AND accepted_volume != 0
    AND ( start_time < '2023-04-01'
    OR start_time > '2023-10-01')
"""
print(pd.read_sql(sql, con))

   avg_ask_summer
0        0.662067
   avg_bid_summer
0        0.767148
   avg_ask_winter
0        0.051921
   avg_bid_winter
0        0.102042


# Total money traded